In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from sklearn.datasets import fetch_openml

In [ ]:
titanic = fetch_openml("titanic", version=1, as_frame=True)

In [ ]:
df = titanic.frame

In [ ]:
print(df.head())

   pclass survived                                             name     sex  \
0       1        1                    Allen, Miss. Elisabeth Walton  female   
1       1        1                   Allison, Master. Hudson Trevor    male   
2       1        0                     Allison, Miss. Helen Loraine  female   
3       1        0             Allison, Mr. Hudson Joshua Creighton    male   
4       1        0  Allison, Mrs. Hudson J C (Bessie Waldo Daniels)  female   

       age  sibsp  parch  ticket      fare    cabin embarked boat   body  \
0  29.0000      0      0   24160  211.3375       B5        S    2    NaN   
1   0.9167      1      2  113781  151.5500  C22 C26        S   11    NaN   
2   2.0000      1      2  113781  151.5500  C22 C26        S  NaN    NaN   
3  30.0000      1      2  113781  151.5500  C22 C26        S  NaN  135.0   
4  25.0000      1      2  113781  151.5500  C22 C26        S  NaN    NaN   

                         home.dest  
0                     St Louis,

In [ ]:
print(df.shape)

(1309, 14)


Список названий столбцов

In [ ]:
print(df.columns.tolist())

['pclass', 'survived', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked', 'boat', 'body', 'home.dest']


Информация о типах данных

In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   pclass     1309 non-null   int64   
 1   survived   1309 non-null   category
 2   name       1309 non-null   object  
 3   sex        1309 non-null   category
 4   age        1046 non-null   float64 
 5   sibsp      1309 non-null   int64   
 6   parch      1309 non-null   int64   
 7   ticket     1309 non-null   object  
 8   fare       1308 non-null   float64 
 9   cabin      295 non-null    object  
 10  embarked   1307 non-null   category
 11  boat       486 non-null    object  
 12  body       121 non-null    float64 
 13  home.dest  745 non-null    object  
dtypes: category(3), float64(3), int64(3), object(5)
memory usage: 116.8+ KB
None


Количество пропусков

In [ ]:
print(df.isnull().sum())

pclass          0
survived        0
name            0
sex             0
age           263
sibsp           0
parch           0
ticket          0
fare            1
cabin        1014
embarked        2
boat          823
body         1188
home.dest     564
dtype: int64


Выделение признаков и целевой переменной

Целевая переменная - survived

In [ ]:
target = "survived"
X = df.drop(columns=[target])
y = df[target]

Первые 5 строк признаков

In [ ]:
print(X.head())

   pclass                                             name     sex      age  \
0       1                    Allen, Miss. Elisabeth Walton  female  29.0000   
1       1                   Allison, Master. Hudson Trevor    male   0.9167   
2       1                     Allison, Miss. Helen Loraine  female   2.0000   
3       1             Allison, Mr. Hudson Joshua Creighton    male  30.0000   
4       1  Allison, Mrs. Hudson J C (Bessie Waldo Daniels)  female  25.0000   

   sibsp  parch  ticket      fare    cabin embarked boat   body  \
0      0      0   24160  211.3375       B5        S    2    NaN   
1      1      2  113781  151.5500  C22 C26        S   11    NaN   
2      1      2  113781  151.5500  C22 C26        S  NaN    NaN   
3      1      2  113781  151.5500  C22 C26        S  NaN  135.0   
4      1      2  113781  151.5500  C22 C26        S  NaN    NaN   

                         home.dest  
0                     St Louis, MO  
1  Montreal, PQ / Chesterville, ON  
2  Montreal

Первые 5 значений целевой переменной y

In [ ]:
print(y.head())

0    1
1    1
2    0
3    0
4    0
Name: survived, dtype: category
Categories (2, object): ['0', '1']


Удаление лишних столбцов:

boat - признак дает утечку информации о целевой переменной

body - номер найденного тела, ответ для класса "не выжил"

home.dest - усложняет кодирование

name, ticket - невысокая информативность

cabin - большое количество пропусков

In [ ]:
cols_to_drop = ["name", "ticket", "cabin", "boat", "body", "home.dest"]

Формируем список столбцов, которые действительно есть в таблице, и удаляем ненужные столбцы из признаков

In [ ]:
existing_cols_to_drop = [col for col in cols_to_drop if col in X.columns]
X = X.drop(columns=existing_cols_to_drop)

Признаки после удаления лишних столбцов

In [ ]:
print(X.columns.tolist())

['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']


Деление признаков на числовые и категориальные

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

Числовые признаки

In [ ]:
print(numeric_features)

['pclass', 'age', 'sibsp', 'parch', 'fare']


Категориальные признаки

In [ ]:
print(categorical_features)

['sex', 'embarked']


Предобработка данных

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

Числовые признаки: заполняем пропуски медианой, масштабируем

Создаем pipeline для числовых признаков

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

Категориальные признаки: заполняем пропуски самым частым значением, кодируем

In [ ]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

Общая схема преобразования для применения разной обработки числовых и категориальных типов данных

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

Разделение на обучающуюся и тестовую выборки

In [ ]:
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    KFold
)

Делим данные на обучающую и тестовую выборки

X_train, y_train - данные, на которых модель будет учиться.

X_test, y_test - данные, на которых будем проверять качество модели.

test_size=0.2 - 20% данных пойдет в тестовую выборку, 80% останется для обучения.

random_state=42 фиксирует случайность, чтобы при каждом запуске получалось одно и то же разбиение.

stratify=y - соотношение классов сохранится и в обучающей, и в тестовой выборке.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Размер обучающей выборки

In [ ]:
print(X_train.shape)

(1047, 7)


Размер тестовой выборки

In [ ]:
print(X_test.shape)

(262, 7)


Исходная модель KNN с произвольным K (метод k ближайших соседей)

In [ ]:
initial_k = 5

initial_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier(n_neighbors=initial_k))
])

Обучаем исходную модель на обучающей выборке.

Метод fit() запоминает обучающие данные и подготавливает все этапы пайплайна:

заполнение пропусков, кодирование, масштабирование и сам классификатор.

predict() - получаем предсказания для объектов из тестовой выборки.

y_pred_initial - это ответы модели (выжил или нет)

In [ ]:
initial_model.fit(X_train, y_train)
y_pred_initial = initial_model.predict(X_test)

Исходная модель KNN

In [ ]:
print(f"\nK={initial_k}")
print("Accuracy:", round(accuracy_score(y_test, y_pred_initial), 4))
print("Precision:", round(precision_score(y_test, y_pred_initial, pos_label='1'), 4))
print("Recall:", round(recall_score(y_test, y_pred_initial, pos_label='1'), 4))
print("F1-score:", round(f1_score(y_test, y_pred_initial, pos_label='1'), 4))


K=5
Accuracy: 0.7977
Precision: 0.7282
Recall: 0.75
F1-score: 0.7389


Классификация

In [ ]:
print(classification_report(y_test, y_pred_initial))

              precision    recall  f1-score   support

           0       0.84      0.83      0.83       162
           1       0.73      0.75      0.74       100

    accuracy                           0.80       262
   macro avg       0.79      0.79      0.79       262
weighted avg       0.80      0.80      0.80       262



Стратегии кросс-валидации (способ более надежно оценить модель, чем одна проверка на одном разбиении)

Стратегия 1: StratifiedKFold

То же что и KFold, но дополнительно сохраняет соотношение классов в каждой части

n_splits=5 - делим на 5 частей.

shuffle=True - перед делением перемешиваем данные.

random_state=42 - фиксируем результат перемешивания.

In [ ]:
cv_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Стратегия 2: KFold

Данные делятся на 5 частей (folds)

4 части идут на обучение, 1 часть - на проверку, так 5 раз пока каждая часть не побывает тестовой

In [ ]:
cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)

Модель для подбора гиперпараметров (настройки модели, которые задаются до обучения)

In [ ]:
search_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier())
])

Сетка параметров

In [ ]:
param_grid = {
    "classifier__n_neighbors": list(range(1, 31)),
    "classifier__weights": ["uniform", "distance"],
    "classifier__metric": ["euclidean", "manhattan"]
}

GridSearchCV + StratifiedKFold (полный перебор параметров по заданной сетке + их проверка)

In [ ]:
grid_search = GridSearchCV(
    estimator=search_model,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv_stratified,
    n_jobs=-1
)

Запуск процедуры гиперпараметров

In [ ]:
grid_search.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['pclass',
                                                                          'age',
                                                                          'sibsp',
                                                                          'parch',
                                                                          'fare']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_f...
                                                                                         ('encoder',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['sex',
                                                                          'embarked'])])),
                                       ('classifier', KNeighborsClassifier())]),
             n_jobs=-1,
             param_grid={'classifier__metric': ['euclidean', 'manhattan'],
                         'classifier__n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8, 9,
                                                     10, 11, 12, 13, 14, 15, 16,
                                                     17, 18, 19, 20, 21, 22, 23,
                                                     24, 25, 26, 27, 28, 29,
                                                     30],
                         'classifier__weights': ['uniform', 'distance']},
             scoring='accuracy')

In [ ]:
best_grid_model = grid_search.best_estimator_
y_pred_grid = best_grid_model.predict(X_test)

Лучшие параметры

In [ ]:
print(grid_search.best_params_)

{'classifier__metric': 'manhattan', 'classifier__n_neighbors': 22, 'classifier__weights': 'uniform'}


Лучшая accuracy на кросс-валидации

In [ ]:
print(round(grid_search.best_score_, 4))

0.7975


Test Accuracy

In [ ]:
print(round(accuracy_score(y_test, y_pred_grid), 4))

0.8244


Test Precision

In [ ]:
print(round(precision_score(y_test, y_pred_grid, pos_label='1'), 4))

0.8068


Test Recall

In [ ]:
print(round(recall_score(y_test, y_pred_grid, pos_label='1'), 4))

0.71


Test F1-score

In [ ]:
print(round(f1_score(y_test, y_pred_grid, pos_label='1'), 4))

0.7553


RandomizedSearchCV + KFold

In [ ]:
random_search = RandomizedSearchCV(
    estimator=search_model,
    param_distributions=param_grid,
    n_iter=15,
    scoring="accuracy",
    cv=cv_kfold,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strategy='median')),
                                                                                               ('scaler',
                                                                                                StandardScaler())]),
                                                                               ['pclass',
                                                                                'age',
                                                                                'sibsp',
                                                                                'parch',
                                                                                'fare']),
                                                                              ('cat',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strategy='most_frequ...
                                                                                                OneHotEncoder(handle_unknown='ignore'))]),
                                                                               ['sex',
                                                                                'embarked'])])),
                                             ('classifier',
                                              KNeighborsClassifier())]),
                   n_iter=15, n_jobs=-1,
                   param_distributions={'classifier__metric': ['euclidean',
                                                               'manhattan'],
                                        'classifier__n_neighbors': [1, 2, 3, 4,
                                                                    5, 6, 7, 8,
                                                                    9, 10, 11,
                                                                    12, 13, 14,
                                                                    15, 16, 17,
                                                                    18, 19, 20,
                                                                    21, 22, 23,
                                                                    24, 25, 26,
                                                                    27, 28, 29,
                                                                    30],
                                        'classifier__weights': ['uniform',
                                                                'distance']},
                   random_state=42, scoring='accuracy')

In [ ]:
best_random_model = random_search.best_estimator_
y_pred_random = best_random_model.predict(X_test)

RandomizedSearchCV + KFold

Лучшие параметры

In [ ]:
print(random_search.best_params_)

{'classifier__weights': 'uniform', 'classifier__n_neighbors': 6, 'classifier__metric': 'euclidean'}


Лучшая accuracy на кросс-валидации

In [ ]:
print(round(random_search.best_score_, 4))

0.7946


Test Accuracy

In [ ]:
print(round(accuracy_score(y_test, y_pred_random), 4))

0.8015


Test Precision

In [ ]:
print(round(precision_score(y_test, y_pred_random, pos_label='1'), 4))

0.7553


Test Recall

In [ ]:
print(round(recall_score(y_test, y_pred_random, pos_label='1'), 4))

0.71


Test F1-score

In [ ]:
print(round(f1_score(y_test, y_pred_random, pos_label='1'), 4))

0.732


Сравнение результатов

In [ ]:
results = pd.DataFrame({
    "Модель": [
        f"KNN (K={initial_k})",
        "GridSearchCV best model",
        "RandomizedSearchCV best model"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_initial),
        accuracy_score(y_test, y_pred_grid),
        accuracy_score(y_test, y_pred_random)
    ],
    "Precision": [
        precision_score(y_test, y_pred_initial, pos_label='1'),
        precision_score(y_test, y_pred_grid, pos_label='1'),
        precision_score(y_test, y_pred_random, pos_label='1')
    ],
    "Recall": [
        recall_score(y_test, y_pred_initial, pos_label='1'),
        recall_score(y_test, y_pred_grid, pos_label='1'),
        recall_score(y_test, y_pred_random, pos_label='1')
    ],
    "F1-score": [
        f1_score(y_test, y_pred_initial, pos_label='1'),
        f1_score(y_test, y_pred_grid, pos_label='1'),
        f1_score(y_test, y_pred_random, pos_label='1')
    ]
})

print(results.round(4))

                          Модель  Accuracy  Precision  Recall  F1-score
0                      KNN (K=5)    0.7977     0.7282    0.75    0.7389
1        GridSearchCV best model    0.8244     0.8068    0.71    0.7553
2  RandomizedSearchCV best model    0.8015     0.7553    0.71    0.7320
